# PPO + LSTM Trading — Kaggle Training Notebook

Notebook này dùng để **train PPO + LSTM** cho dự án trading trên Kaggle.

| Thành phần | Vai trò |
|-----------|---------|
| `src/environment/trading_env.py` | Môi trường Gymnasium cho trading |
| `src/models/lstm.py` | Mạng `PPOLSTMActorCritic` (LSTM + Actor-Critic) |
| `src/agents/ppo_agent.py` | `PPOAgent` + `RolloutBuffer` (thu thập rollout, update PPO) |
| `src/training/PPO.py` | Training loop chính (`train_ppo`) |
| `Conf/ppo_conf.yaml` | Cấu hình hyperparameter (dễ chỉnh trên Kaggle) |

Kịch bản chạy:
1. Cài đặt thư viện cần thiết
2. Clone repo từ GitHub vào `/kaggle/working/repo`
3. Đọc config từ `Conf/ppo_conf.yaml`
4. Gọi `train_ppo(config)` để train và log kết quả bởi `TrainingLogger`.

In [ ]:
# 1. Cài đặt thư viện cần thiết (nếu thiếu)

import subprocess
import sys

# Cài gymnasium (thường chưa có sẵn trên Kaggle)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium"], check=True)

print("Thư viện đã sẵn sàng.")

## 2. Clone GitHub Repo vào `/kaggle/working/repo`

Sử dụng Kaggle Secret `GH_TOKEN` để clone repo chứa mã nguồn PPO + LSTM.

- Cần tạo secret `GH_TOKEN` trong Kaggle (Settings → Secrets)
- Cập nhật biến `GITHUB_REPO` bên dưới thành repo chứa project này (ví dụ: `"username/RL-Trading"`).

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import subprocess
import sys

# =============================================================
# CẬP NHẬT TÊN REPO TẠI ĐÂY
# =============================================================
GITHUB_REPO  = "username/RL-Trading"  # TODO: thay bằng repo thật trên GitHub
BRANCH       = "main"
# =============================================================

user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GH_TOKEN")

CLONE_DIR   = Path("/kaggle/working/repo")
WORKING_DIR = Path("/kaggle/working")

if CLONE_DIR.exists():
    result = subprocess.run(
        ["git", "-C", str(CLONE_DIR), "pull"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
else:
    repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    result = subprocess.run(
        ["git", "clone", "--depth=1", "-b", BRANCH, repo_url, str(CLONE_DIR)],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )

print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("Git clone/pull thất bại. Kiểm tra lại GH_TOKEN và GITHUB_REPO.")

PROJECT_ROOT = CLONE_DIR
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

## 3. Đọc cấu hình PPO từ `Conf/ppo_conf.yaml`

File config nằm trong repo vừa clone, dùng để điều chỉnh nhanh hyperparameter khi train trên Kaggle.

In [ ]:
import yaml
from pprint import pprint

CONF_PATH = PROJECT_ROOT / "Conf" / "ppo_conf.yaml"

with open(CONF_PATH, "r", encoding="utf-8") as f:
    ppo_cfg = yaml.safe_load(f)

# -------------------------------------------------------------
# OVERRIDE đường dẫn data khi chạy trên Kaggle
# -------------------------------------------------------------
# Dataset Kaggle: /kaggle/input/datasets/phctontrn/dataset-trading-rl
# Giả định các file ACB.csv, BID.csv,... nằm trực tiếp trong thư mục này.
ppo_cfg["data_path"] = "/kaggle/input/datasets/phctontrn/dataset-trading-rl"

print("Đường dẫn config:", CONF_PATH)
print("Data path (Kaggle):", ppo_cfg["data_path"])
print("\nCấu hình PPO (rút gọn):")
keys_to_show = [
    "tickers", "features", "window_size",
    "train_ratio", "val_ratio", "test_ratio",
    "initial_balance", "fee_rate",
    "learning_rate", "n_steps", "batch_size", "total_timesteps",
]
summary = {k: ppo_cfg.get(k) for k in keys_to_show}
pprint(summary)

## 4. Chạy training PPO + LSTM

Cell dưới sẽ gọi trực tiếp `train_ppo(config)` trong `src/training/PPO.py`.

- Logger sẽ tự tạo `run_id` và ghi log vào `results/runs/<run_id>` bên trong repo
- Khi muốn thử hyperparameters khác, chỉ cần chỉnh `Conf/ppo_conf.yaml` rồi chạy lại cell này.

In [ ]:
import sys

# Đảm bảo PROJECT_ROOT đã nằm trong sys.path từ cell clone repo
sys.path.insert(0, str(PROJECT_ROOT))

from src.training.PPO import train_ppo

# Có thể override thêm một số hyperparameters ngay tại notebook nếu muốn, ví dụ:
# ppo_cfg["total_timesteps"] = 200_000
# ppo_cfg["learning_rate"] = 2e-4

agent, test_metrics = train_ppo(ppo_cfg)

print("\nKết quả test trung bình (tóm tắt):")
for k, v in sorted(test_metrics.items()):
    print(f"  {k}: {v}")